# 本地 Vision + LangChain 工具（mock 实现）

与 `19_local_vision_minimal.ipynb` 同一套 LM Studio / OpenAI 兼容端点，本文件**独立运行**，不修改 19 原文。

- 图片：读本地文件 → base64 `data:` URL（逻辑与 19 一致）。
- 工具：用 `@tool` 注册，**函数体为 mock 固定返回**，可日后替换为真实 API。
- 模型需支持 **function / tool calling**；若不报 `tool_calls`，最后会回退为仅打印模型直接回复。

In [3]:
%pip install -q "langchain-core>=0.3" "langchain-openai>=0.2"


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import base64
import traceback
from pathlib import Path
from typing import Any
from uuid import UUID

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

BASE_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
MODEL = "qwen/qwen3-vl-4b"

IMAGE_NAME = "微信图片_20260504122723_232_510.jpg"


def resolve_local_image(name: str) -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for rel in (
            base / "notebooks" / "19_image" / name,
            base / "19_image" / name,
        ):
            if rel.is_file():
                return rel.resolve()
    raise FileNotFoundError(
        f"找不到 {name}。当前 cwd: {here}。请确认文件在 notebooks/19_image/ 下或改为绝对路径。"
    )


IMAGE_PATH = resolve_local_image(IMAGE_NAME)
suffix = IMAGE_PATH.suffix.lower()
mime = (
    "image/jpeg"
    if suffix in {".jpg", ".jpeg"}
    else "image/png"
    if suffix == ".png"
    else "application/octet-stream"
)
b64 = base64.standard_b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
image_url = f"data:{mime};base64,{b64}"
print("使用图片:", IMAGE_PATH)


@tool
def get_mock_weather(region: str) -> str:
    """查询某地区的天气概况。当前为占位实现，不真实请求气象服务。"""
    return f"[MOCK 天气] {region}: 晴间多云，10–15°C，南风 2 级"


@tool
def lookup_scene_tags(scene_hint: str) -> str:
    """根据画面文字描述补充结构化标签。当前为占位，不真实做图像检索。"""
    return f"[MOCK 标签] hint={scene_hint!r} -> 高原, 山地, 人文设施"


tools = [get_mock_weather, lookup_scene_tags]
tools_by_name = {t.name: t for t in tools}


def _is_notebook_noise(path: str) -> bool:
    p = path.replace("\\", "/")
    return any(
        x in p
        for x in (
            "ipykernel",
            "IPython",
            "tornado/platform",
            "asyncio/base_events",
            "concurrent/futures",
        )
    )


def _langchain_openai_code_path(max_frames: int = 20) -> str:
    """从当前点向上，只保留与 LLM 请求相关的库内调用帧（非完整 Python 栈）。"""
    lines: list[str] = []
    for fr in traceback.extract_stack()[:-1]:
        if _is_notebook_noise(fr.filename):
            continue
        p = fr.filename.replace("\\", "/")
        if not any(
            s in p
            for s in (
                "langchain_core",
                "langchain_openai",
                "langchain_community",
                "/openai/",
            )
        ):
            continue
        lines.append(f"  {p}:{fr.lineno}  {fr.name}")
    if not lines:
        return "  (未匹配到 langchain/openai 相关帧；可把上方判断放宽)"
    return "\n".join(lines[-max_frames:])


class LlmCallChainPrinter(BaseCallbackHandler):
    """打印 LangChain Runnable 父子关系上的「到 ChatModel 为止」的链 + 库内代码路径。"""

    def __init__(self) -> None:
        self._run_label: dict[UUID, str] = {}
        self._parent: dict[UUID, UUID | None] = {}

    @staticmethod
    def _label(serialized: dict[str, Any], **kwargs: Any) -> str:
        if kwargs.get("name"):
            return str(kwargs["name"])
        if serialized.get("name"):
            return str(serialized["name"])
        id_ = serialized.get("id")
        if isinstance(id_, list) and id_:
            return str(id_[-1])
        return str(serialized.get("type", "?"))

    def on_chain_start(
        self,
        serialized: dict[str, Any],
        inputs: dict[str, Any],
        *,
        run_id: UUID,
        parent_run_id: UUID | None = None,
        **kwargs: Any,
    ) -> None:
        self._run_label[run_id] = self._label(serialized, **kwargs)
        self._parent[run_id] = parent_run_id

    def on_chat_model_start(
        self,
        serialized: dict[str, Any],
        messages: list[list[Any]],
        *,
        run_id: UUID,
        parent_run_id: UUID | None = None,
        **kwargs: Any,
    ) -> None:
        self._run_label[run_id] = self._label(serialized, **kwargs)
        self._parent[run_id] = parent_run_id
        path_names: list[str] = []
        cur: UUID | None = run_id
        seen: set[UUID] = set()
        while cur is not None and cur not in seen:
            seen.add(cur)
            path_names.append(self._run_label.get(cur, f"?{cur!s}"))
            cur = self._parent.get(cur)
        path_names.reverse()
        print(f"\n{'=' * 60}\n[LangChain → LLM 调用链]  {' -> '.join(path_names)}")
        print(f"[run_id] {run_id}  [parent_run_id] {parent_run_id}")
        print("[库内代码路径 · 自外向内至当前回调](langchain / openai)\n" + _langchain_openai_code_path())
        print("=" * 60)


llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)
llm_trace = LlmCallChainPrinter()

user = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "先看图，用一句话概括场景；若合适，调用 lookup_scene_tags 传入简短中文描述；"
                "再调用 get_mock_weather，region 填你推测的大致地理区域（如「青藏高原」）。"
                "最后用中文汇总工具返回。"
            ),
        },
        {"type": "image_url", "image_url": {"url": image_url}},
    ]
)

messages: list = [user]
ai_msg: AIMessage = llm_with_tools.invoke(
    messages, config={"callbacks": [llm_trace]}
)
messages.append(ai_msg)

if getattr(ai_msg, "tool_calls", None):
    for tc in ai_msg.tool_calls:
        name = tc["name"]
        out = tools_by_name[name].invoke(tc["args"])
        messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
    final = llm_with_tools.invoke(
        messages, config={"callbacks": [llm_trace]}
    )
    print(final.content)
else:
    print("(模型未发起 tool_calls，直接回复：)", ai_msg.content)

使用图片: /Users/ludan/Downloads/git-repo/chanon-data-lab/notebooks/19_image/微信图片_20260504122723_232_510.jpg

[stack] 首次 llm_with_tools.invoke（首轮，含 vision + bind_tools）


  File "/Users/ludan/Downloads/git-repo/chanon-data-lab/.venv_py312/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
    handle._run()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "/Users/ludan/Downloads/git-repo/chanon-data-lab/.venv_py312/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 621, in shell_main
    await self.dispatch_shell(msg, subshell_id=subshell_id)
  File "/Users/ludan/Downloads/git-repo/chanon-data-lab/.venv_py312


[stack] 执行工具: lookup_scene_tags，args={'scene_hint': '雪山、草原、天空、树木'}

[stack] 执行工具: get_mock_weather，args={'region': '青藏高原'}

[stack] 第二次 llm_with_tools.invoke（已附 ToolMessage，生成最终回复）


  File "/Users/ludan/Downloads/git-repo/chanon-data-lab/.venv_py312/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once
    handle._run()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "/Users/ludan/Downloads/git-repo/chanon-data-lab/.venv_py312/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 621, in shell_main
    await self.dispatch_shell(msg, subshell_id=subshell_id)
  File "/Users/ludan/Downloads/git-repo/chanon-data-lab/.venv_py312

该场景为青藏高原的雪山草原风光，天空晴间多云，气温10–15°C，风力2级。
